# SAM3 for FiftyOne - Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harpreetsahota204/sam3_images/blob/main/sam3_quickstart.ipynb)

Complete guide to using Meta's SAM3 (Segment Anything Model 3) in FiftyOne.

**Features:**
- 🎯 Concept Segmentation (text prompts)
- 🖱️ Visual Segmentation (interactive prompts)
- 🤖 Automatic Segmentation (no prompts)
- 🔍 Visual Embeddings (1024-dim)


## 1. Installation

SAM3 requires transformers from source (not yet on PyPI):


In [26]:
# Authenticate with Hugging Face for SAM3 model access
# Load credentials from CV workspace secrets
import pathlib
import yaml
from huggingface_hub import login

secrets_path = pathlib.Path.cwd().parent / "cvmgr" / "configs" / "secrets.yaml"

if secrets_path.exists():
    with secrets_path.open('r') as file:
        secrets_yaml = yaml.safe_load(file)
    
    login(secrets_yaml["huggingface"]["token"])
    print("✅ Authenticated with Hugging Face")
else:
    print("⚠️ secrets.yaml not found. Please set up Hugging Face authentication:")
    print("1. Create cvmgr/configs/secrets.yaml")
    print("2. Add your HF token: huggingface: token: 'your_token_here'")
    # Fallback to manual login
    # login()  # Uncomment if you want to login manually

✅ Authenticated with Hugging Face


## 2. Setup


In [27]:
import fiftyone.zoo as foz
import fiftyone as fo

# Delete dataset if it exists
dataset_name = "open-images-v7-validation-segmentations"
if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)
    print(f"✅ Deleted existing dataset: {dataset_name}")

dataset = foz.load_zoo_dataset(
    "open-images-v7",
    splits=["validation"],  # Use validation split for consistent evaluation
    label_types=["segmentations"],  # Focus on segmentation masks
    classes=["Door handle"],  # Specific class for concept segmentation
    max_samples=100,  # Limit for efficient evaluation
    shuffle=True,
    seed=42  # Reproducible sampling
)

Only found 26 (<100) samples matching your requirements
Necessary images already downloaded
Existing download of split 'validation' is sufficient
Loading existing dataset 'open-images-v7-validation-100'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


In [28]:
import fiftyone as fo
import fiftyone.brain as fob
import torch

# Register SAM3 zoo model
foz.register_zoo_model_source(
    "https://github.com/harpreetsahota204/sam3_images"
)

# Load model
model = foz.load_zoo_model("facebook/sam3")

print(f"✅ Loaded {len(dataset)} samples")

device=torch.cuda.is_available()
print(device
)

✅ Loaded 26 samples
True


## 3. Visual Embeddings

Extract 1024-dim embeddings for similarity search:


In [29]:
# Compute embeddings
model.pooling_strategy = "max"  # or "mean", "cls"

dataset.compute_embeddings(
    model,
    embeddings_field="sam_embeddings",
    batch_size=16,
    num_workers=8
)

print("✅ Embeddings computed")


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable 

TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true

 100% |███████████████████| 26/26 [22.4s elapsed, 0s remaining, 1.2 samples/s]   

TOKENIZERS_PARALLELISM=(true | false)


 100% |███████████████████| 26/26 [22.6s elapsed, 0s remaining, 1.2 samples/s]   
✅ Embeddings computed


In [30]:
# Visualize with UMAP
fob.compute_visualization(
    dataset,
    method="umap",
    brain_key="sam_viz",
    embeddings="sam_embeddings",
    num_dims=2
)

print("✅ UMAP visualization ready")


Generating visualization...
UMAP( verbose=True)
Mon Dec  8 15:22:10 2025 Construct fuzzy simplicial set
Mon Dec  8 15:22:10 2025 Finding Nearest Neighbors
Mon Dec  8 15:22:10 2025 Finished Nearest Neighbor Search
Mon Dec  8 15:22:10 2025 Construct embedding


Epochs completed:   0%|            0/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs
	completed  150  /  500 epochs
	completed  200  /  500 epochs
	completed  250  /  500 epochs
	completed  300  /  500 epochs
	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs
Mon Dec  8 15:22:14 2025 Finished embedding
✅ UMAP visualization ready


## 4. Concept Segmentation

Find ALL instances matching text concepts:


In [31]:
# Single concept
model.operation = "concept_segmentation"
model.prompt = "Door handle"
model.threshold = 0.5
model.mask_threshold = 0.5

dataset.apply_model(
    model,
    label_field="Door_handle_masks",
    batch_size=16,
    num_workers=8
)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

 100% |███████████████████| 26/26 [25.4s elapsed, 0s remaining, 1.0 samples/s]   


## 8. Explore Results


In [33]:
import fiftyone as fo
import fiftyone.zoo as foz

print(dataset)

results = dataset.evaluate_detections(
    "Door_handle_masks",
    gt_field="ground_truth",
    method="open-images",
)

print(results.mAP())
# 0.599
plot = results.plot_pr_curves(classes=["Door handle"])
plot.show()

Name:        open-images-v7-validation-100
Media type:  image
Num samples: 26
Persistent:  False
Tags:        []
Sample fields:
    id:                fiftyone.core.fields.ObjectIdField
    filepath:          fiftyone.core.fields.StringField
    tags:              fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:          fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:        fiftyone.core.fields.DateTimeField
    last_modified_at:  fiftyone.core.fields.DateTimeField
    ground_truth:      fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    sam_embeddings:    fiftyone.core.fields.VectorField
    door_handle_masks: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    predictions:       fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections)
    Door_handle_masks: fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.labels.Detections

FigureWidget({
    'data': [{'customdata': {'bdata': ('AAAAAAAA8D8AAABgOLruPwAAAGA4uu' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAA=='),
                             'dtype': 'f8'},
              'hovertemplate': ('<b>class: %{text}</b><br>recal' ... 'customdata:.3f}<extra></extra>'),
              'line': {'color': '#FF6D04'},
              'mode': 'lines',
              'name': 'Door handle (AP = 0.539)',
              'text': array(['Door handle', 'Door handle', 'Door handle', 'Door handle',
                             'Door handle', 'Door handle', 'Door handle', 'Door handle',
                             'Door handle', 'Door handle', 'Door handle', 'Door handle',
                             'Door handle', 'Door handle', 'Door handle', 'Door handle',
                             'Door handle', 'Door handle', 'Door handle', 'Door handle',
                             'Door handle', 'Door handle', 'Door handle', 'Door handle',
                             'Door handle', 'Door handle', 'Door h

In [34]:
import random
import fiftyone as fo



# Load subset of ActivityNet 200

classes = ["Door handle"]


# Generate some fake predictions for this example

random.seed(51)

dataset.clone_sample_field("ground_truth", "predictions")

for sample in dataset:

    for det in sample.predictions.detections:
        det.support[0] += random.randint(-10,10)

        det.support[1] += random.randint(-10,10)

        det.support[0] = max(det.support[0], 1)

        det.support[1] = max(det.support[1], det.support[0] + 1)

        det.confidence = random.random()

        det.label = random.choice(classes)


    sample.save()


# Performs an IoU sweep so that mAP and PR curves can be computed

results = dataset.evaluate_detections(

    "predictions",

    gt_field="ground_truth",

    eval_key="eval",

    compute_mAP=True,

)


print(results.mAP())

# 0.367


plot = results.plot_pr_curves(classes=classes)

plot.show()

ValueError: Sample field 'predictions' already exists

## Summary

You've now:
- ✅ Computed visual embeddings for similarity search
- ✅ Found objects using text prompts (concept segmentation)
- ✅ Refined detections with precise masks (visual segmentation)
- ✅ Generated all possible masks (automatic segmentation)

**Next Steps:**
- Try different prompts for concept segmentation
- Adjust thresholds for more/fewer detections
- Experiment with automatic segmentation parameters

**Documentation:** https://github.com/harpreetsahota204/sam3_images
